# DeepFakeShield — Model Training (real datasets)
FaceForensics++ C23 extracted faces + fake-vs-real speech.
Author: vtangri | Repo: https://github.com/vtangri/DeepFakeShield

In [ ]:
# ── Cell 1: Environment — install a torch build that supports this GPU ──
# IMPORTANT: torch must NOT be imported before the (possible) reinstall below,
# or the already-loaded module would stay in memory for the whole session.
import sys, os, json, subprocess

print('Python:', sys.version)

def sh(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('pip failed:', r.stderr[-800:])
    return r.returncode == 0

def gpu_arch():
    """Read the GPU's compute capability via nvidia-smi, without importing torch."""
    try:
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=60)
        if out.returncode != 0 or not out.stdout.strip():
            return None, None
        name, cap = [x.strip() for x in out.stdout.strip().splitlines()[0].split(',')]
        return name, int(float(cap) * 10)   # '6.0' -> 60
    except Exception as e:
        print('nvidia-smi unavailable:', e)
        return None, None

GPU_NAME, SM = gpu_arch()
print(f'Detected GPU: {GPU_NAME} (sm_{SM})' if GPU_NAME else 'No GPU detected')

# Kaggle's default image ships torch 2.10+cu128, whose wheels start at sm_70.
# A P100 is sm_60, so downgrade to the last build that still shipped sm_60.
if SM is not None and SM < 70:
    print(f'\nsm_{SM} predates the stock torch build. Installing torch 2.5.1+cu121...')
    ok = sh('install', '-q',
            'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
            '--index-url', 'https://download.pytorch.org/whl/cu121')
    print('torch downgrade:', 'OK' if ok else 'FAILED (will fall back to CPU)')

for pkg in ['scikit-learn', 'matplotlib', 'seaborn', 'tqdm', 'opencv-python-headless']:
    print(('OK: ' if sh('install', '-q', pkg) else 'WARNING: failed ') + pkg)

# Only now is it safe to import torch.
import torch
print('\nPyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

def pick_device():
    """Use CUDA only if this torch build actually has kernels for the GPU.

    A mismatch here is not caught by torch.cuda.is_available() -- it surfaces
    much later as 'no kernel image is available for execution on the device'.
    """
    if not torch.cuda.is_available():
        print('No CUDA device -> CPU')
        return 'cpu'
    cap = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    print('GPU:', torch.cuda.get_device_name(0))
    print('Device capability: sm_%d%d | torch arch list: %s' % (cap[0], cap[1], supported))
    if f'sm_{cap[0]}{cap[1]}' not in supported:
        print(f'WARNING: {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]}) is not '
              f'supported by torch {torch.__version__}. Falling back to CPU.')
        return 'cpu'
    # Prove a real kernel launches before committing to CUDA.
    try:
        (torch.randn(64, 64, device='cuda') @ torch.randn(64, 64, device='cuda')).sum().item()
    except Exception as e:
        print(f'WARNING: CUDA smoke test failed ({type(e).__name__}: {e}). Falling back to CPU.')
        return 'cpu'
    return 'cuda'

DEVICE = pick_device()
print('Using device:', DEVICE)
print('Environment ready.')


In [ ]:
# ── Cell 2: Locate real datasets ─────────────────────────────────────
import os, sys, json
from pathlib import Path
import torch

MODEL_DIR = Path('/kaggle/working/models')
EVAL_DIR  = Path('/kaggle/working/evaluation')
for d in [MODEL_DIR, EVAL_DIR]: d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224          # ViT-B/16 input
SR       = 16000
MAX_SEC  = 3.0

def find_root(*candidates, must_contain=None):
    """Return the first existing dataset path, else raise with a listing."""
    for c in candidates:
        p = Path(c)
        if p.exists() and (must_contain is None or (p / must_contain).exists()):
            return p
    avail = []
    base = Path('/kaggle/input')
    if base.exists():
        for d in sorted(base.iterdir()):
            avail.append(f'  {d}')
            for sub in sorted(d.iterdir())[:6]:
                avail.append(f'    {sub.name}')
    raise FileNotFoundError(
        'None of these dataset paths exist:\n  ' + '\n  '.join(map(str, candidates))
        + '\nAttach the dataset in kernel-metadata.json dataset_sources.'
        + '\nAvailable under /kaggle/input:\n' + '\n'.join(avail))

VIDEO_ROOT = find_root(
    '/kaggle/input/faceforensics-c23-extracted-faces-100k/dataset_processed_split',
    must_contain='dataset_manifest.csv')
AUDIO_ROOT = find_root(
    '/kaggle/input/deepfake-audio-dataset-fake-vs-real-speech/deepfake_audio_dataset_jay15k',
    '/kaggle/input/deepfake-audio-dataset-fake-vs-real-speech')

print('VIDEO_ROOT:', VIDEO_ROOT)
print('AUDIO_ROOT:', AUDIO_ROOT, '->', sorted(p.name for p in AUDIO_ROOT.iterdir() if p.is_dir()))

# Cap samples per class so a run finishes in reasonable time. CPU is far
# too slow for the full 182k-frame set, so shrink hard when GPU is absent.
if DEVICE == 'cuda':
    CAPS = {'train': 6000, 'val': 1500, 'test': 1500}
    VIDEO_EPOCHS, AUDIO_EPOCHS = 6, 12
else:
    print('\n*** WARNING: no usable GPU. Using a small subset so the run finishes.')
    print('*** Results will be weak. Set Accelerator=GPU T4 x2 in the Kaggle UI.\n')
    CAPS = {'train': 400, 'val': 100, 'test': 100}
    VIDEO_EPOCHS, AUDIO_EPOCHS = 2, 4

print('Per-class caps:', CAPS)


In [ ]:
# ── Cell 3: Dataset classes over the real data ───────────────────────
import csv, random
import numpy as np
import torch, torchaudio
from PIL import Image
from torch.utils.data import Dataset, DataLoader

random.seed(0)

IMAGENET_MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
IMAGENET_STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

def _balance(items, cap):
    """Downsample each class to the same size (<= cap). items: list of (path,label)."""
    by = {0: [x for x in items if x[1]==0], 1: [x for x in items if x[1]==1]}
    n = min(len(by[0]), len(by[1]), cap)
    if n == 0:
        raise ValueError(f'Cannot balance: real={len(by[0])} fake={len(by[1])}')
    out = []
    for lbl in (0,1):
        random.shuffle(by[lbl]); out += by[lbl][:n]
    random.shuffle(out)
    return out

class VideoFramesDS(Dataset):
    """FaceForensics++ extracted faces, indexed via dataset_manifest.csv.

    Layout: {root}/{split}/{fake_type}/{filename}. label REAL->0, FAKE->1.
    """
    _manifest = None

    @classmethod
    def manifest(cls, root):
        if cls._manifest is None:
            with open(Path(root)/'dataset_manifest.csv') as f:
                cls._manifest = list(csv.DictReader(f))
        return cls._manifest

    def __init__(self, root, split, cap):
        self.root = Path(root)
        rows = [r for r in self.manifest(root) if r['split'] == split]
        items = [(self.root/split/r['fake_type']/r['filename'],
                  0 if r['label'] == 'REAL' else 1) for r in rows]
        self.samples = _balance(items, cap)
        print(f'VideoFramesDS [{split}]: {len(self.samples)} samples '
              f'({sum(1 for s in self.samples if s[1]==0)} real / '
              f'{sum(1 for s in self.samples if s[1]==1)} fake)')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            t = torch.from_numpy(np.array(img)).permute(2,0,1).float() / 255.0
        except Exception as e:
            print(f'WARN image load {path}: {e}')
            t = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        return (t - IMAGENET_MEAN) / IMAGENET_STD, torch.tensor(lbl, dtype=torch.float32)

class RealAudioDS(Dataset):
    """Fake-vs-real speech laid out as {root}/real/*.wav and {root}/fake/*.wav.

    The dataset ships no split, so carve one deterministically per class.
    """
    _pool = None

    @classmethod
    def pool(cls, root):
        if cls._pool is None:
            root = Path(root)
            def grab(name, lbl):
                d = next((c for c in root.iterdir()
                          if c.is_dir() and c.name.lower() == name), None)
                if d is None:
                    raise FileNotFoundError(
                        f"No '{name}' dir under {root}; found "
                        f"{[c.name for c in root.iterdir() if c.is_dir()]}")
                return [(p, lbl) for p in sorted(d.glob('*.wav'))]
            real, fake = grab('real', 0), grab('fake', 1)
            random.shuffle(real); random.shuffle(fake)
            cls._pool = {}
            # 70/15/15 per class, split before any capping
            for lbl, files in ((0, real), (1, fake)):
                n = len(files); a, b = int(0.7*n), int(0.85*n)
                cls._pool.setdefault('train', []).extend(files[:a])
                cls._pool.setdefault('val',   []).extend(files[a:b])
                cls._pool.setdefault('test',  []).extend(files[b:])
            print('Audio pool sizes:', {k: len(v) for k, v in cls._pool.items()})
        return cls._pool

    def __init__(self, root, split, cap):
        self.max_n = int(SR * MAX_SEC)
        self.samples = _balance(self.pool(root)[split], cap)
        print(f'RealAudioDS [{split}]: {len(self.samples)} samples '
              f'({sum(1 for s in self.samples if s[1]==0)} real / '
              f'{sum(1 for s in self.samples if s[1]==1)} fake)')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        try:
            w, sr = torchaudio.load(str(path))
            if sr != SR:
                w = torchaudio.transforms.Resample(sr, SR)(w)
            if w.shape[0] > 1: w = w.mean(0, keepdim=True)
            if w.shape[1] > self.max_n: w = w[:, :self.max_n]
            else: w = torch.nn.functional.pad(w, (0, self.max_n - w.shape[1]))
        except Exception as e:
            print(f'WARN audio load {path}: {e}')
            w = torch.zeros(1, self.max_n)
        return w.squeeze(0), torch.tensor(lbl, dtype=torch.float32)

def make_loaders(DS, root, bs, nw=2):
    pm = torch.cuda.is_available()
    mk = lambda sp, sh: DataLoader(DS(root, sp, CAPS[sp]), bs, shuffle=sh,
                                   num_workers=nw, pin_memory=pm)
    return mk('train', True), mk('val', False), mk('test', False)

print('Dataset classes defined.')


In [ ]:
# ── Cell 4: Model architectures ──────────────────────────────────────
import torch.nn as nn
import torchaudio
from torchvision.models import vit_b_16, ViT_B_16_Weights

def build_video_model():
    # Pretrained weights require internet (phone-verified Kaggle account).
    # Fall back to random init so the run still completes offline.
    try:
        m = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        print('Video backbone: ImageNet-pretrained ViT-B/16')
    except Exception as e:
        print(f'WARNING: could not fetch pretrained ViT weights ({type(e).__name__}: {e}).')
        print('Falling back to randomly-initialised ViT-B/16.')
        m = vit_b_16(weights=None)
    m.heads = nn.Sequential(
        nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 1),   nn.Sigmoid()
    )
    return m

class AudioModel(nn.Module):
    def __init__(self, sr=16000):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=1024, hop_length=256, n_mels=80)
        self.features = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(256*16, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256,1), nn.Sigmoid()
        )
    def forward(self, x):
        m = self.mel(x).unsqueeze(1)
        m = (m - m.mean()) / (m.std() + 1e-8)
        return self.head(self.features(m))

print('Models defined.')

In [ ]:
# ── Cell 5: Training helpers ─────────────────────────────────────────
import torch
from tqdm import tqdm   # plain tqdm, not tqdm.notebook — works everywhere

def run_epoch(model, loader, crit, opt, device, train=True):
    model.train(train)
    loss_sum = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    preds_all, labels_all = [], []
    with ctx:
        for x, y in tqdm(loader, desc='train' if train else 'eval', leave=False):
            x, y = x.to(device), y.to(device).unsqueeze(1)
            out = model(x)
            loss = crit(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item()
            correct  += ((out > 0.5).float() == y).sum().item()
            total    += y.size(0)
            preds_all.extend(out.detach().cpu().numpy().flatten())
            labels_all.extend(y.detach().cpu().numpy().flatten())
    return loss_sum/len(loader), correct/total, preds_all, labels_all

def train_model(model, tr, vl, te, device, epochs, save_path, name, lr=1e-3):
    import torch.optim as optim
    crit = torch.nn.BCELoss()
    opt  = optim.Adam(model.parameters(), lr=lr)
    sched= optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3)
    best_acc, results = 0, []
    print(f'\n=== Training {name} for {epochs} epochs on {device} (lr={lr}) ===')
    for ep in range(1, epochs+1):
        tl, ta, _, _ = run_epoch(model, tr, crit, opt, device, train=True)
        vl2, va, _, _= run_epoch(model, vl, crit, opt, device, train=False)
        sched.step(vl2)
        results.append({'epoch':ep,'train_loss':tl,'train_acc':ta,'val_loss':vl2,'val_acc':va})
        print(f'  Ep {ep:02d}/{epochs} | train loss={tl:.4f} acc={ta:.4f} | val loss={vl2:.4f} acc={va:.4f}')
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), save_path)
            print(f'    -> Saved best ({save_path.name}, val_acc={va:.4f})')
    # Test
    model.load_state_dict(torch.load(save_path, map_location=device))
    _, test_acc, test_preds, test_labels = run_epoch(model, te, crit, opt, device, train=False)
    print(f'  Test accuracy: {test_acc:.4f}')
    return test_acc, test_preds, test_labels, results

print('Training helpers defined.')

In [ ]:
# ── Cell 6: Train Audio Spoof Model on real speech ───────────────────
device = torch.device(DEVICE)

audio_tr, audio_vl, audio_te = make_loaders(RealAudioDS, AUDIO_ROOT, bs=32)

audio_model = AudioModel().to(device)
audio_path  = MODEL_DIR / 'audio_spoof_final.pt'

audio_test_acc, audio_preds, audio_labels, audio_history = train_model(
    audio_model, audio_tr, audio_vl, audio_te,
    device, epochs=AUDIO_EPOCHS, save_path=audio_path, name='AudioSpoofCNN'
)
print(f'Audio model saved to {audio_path}')


In [ ]:
# ── Cell 7: Train Video Forensics Model (ViT-B/16) on FF++ ───────────
video_tr, video_vl, video_te = make_loaders(VideoFramesDS, VIDEO_ROOT, bs=32)

video_model = build_video_model().to(device)
video_path  = MODEL_DIR / 'video_forensics_final.pt'

video_test_acc, video_preds, video_labels, video_history = train_model(
    video_model, video_tr, video_vl, video_te,
    device, epochs=VIDEO_EPOCHS, save_path=video_path, name='VideoViT-B16', lr=1e-4
)
print(f'Video model saved to {video_path}')


In [ ]:
# ── Cell 8: Plot & save evaluation charts ────────────────────────────
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — works in all environments
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

def save_eval_charts(name, preds, labels, out_dir):
    labels_bin = [int(l) for l in labels]
    preds_bin  = [1 if p > 0.5 else 0 for p in preds]
    # ROC
    fpr, tpr, _ = roc_curve(labels_bin, preds)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(fpr, tpr, lw=2, label=f'ROC AUC = {roc_auc:.4f}')
    ax.plot([0,1],[0,1],'k--')
    ax.set(xlabel='FPR', ylabel='TPR', title=f'{name} ROC Curve')
    ax.legend(); fig.tight_layout()
    fig.savefig(out_dir/f'{name}_roc.png', dpi=100); plt.close(fig)
    # Confusion matrix
    cm = confusion_matrix(labels_bin, preds_bin)
    fig, ax = plt.subplots(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real','Fake'], yticklabels=['Real','Fake'], ax=ax)
    ax.set_title(f'{name} Confusion Matrix'); fig.tight_layout()
    fig.savefig(out_dir/f'{name}_cm.png', dpi=100); plt.close(fig)
    # Classification report
    report = classification_report(labels_bin, preds_bin, target_names=['Real','Fake'])
    (out_dir/f'{name}_report.txt').write_text(report)
    # Metrics JSON
    with open(out_dir/f'{name}_metrics.json','w') as f:
        json.dump({'roc_auc': roc_auc, 'test_accuracy': float(sum(p==l for p,l in zip(preds_bin,labels_bin)))/len(labels_bin)}, f, indent=2)
    print(f'{name}: AUC={roc_auc:.4f}')
    print(report)

save_eval_charts('audio', audio_preds, audio_labels, EVAL_DIR)
save_eval_charts('video', video_preds, video_labels, EVAL_DIR)
print('All evaluation charts saved.')

In [ ]:
# ── Cell 9: Summary of all output files ──────────────────────────────
import os
print('=== Trained Models ===')
for f in sorted(MODEL_DIR.glob('*.pt')):
    print(f'  {f.name:45s}  {f.stat().st_size/1024/1024:.1f} MB')

print('\n=== Evaluation Outputs ===')
for f in sorted(EVAL_DIR.iterdir()):
    print(f'  {f.name}')

print('\n=== Final Accuracies ===')
print(f'  Audio test accuracy : {audio_test_acc:.4f}')
print(f'  Video test accuracy : {video_test_acc:.4f}')
print('\nTraining complete!')